# ViT Paper Replication — Colab Verification

Trains both models and prints final test accuracy.

> **Before running:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Clone repo
!git clone https://github.com/Roopesh-BR/vit-paper-replication.git
%cd vit-paper-replication

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q
!pip install -e . -q

In [ ]:
# Confirm GPU
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Train custom ViT from scratch (10 epochs, ~15 min on T4)
!python train.py --model custom --epochs 10 --lr 3e-3 --device cuda

In [ ]:
# Fine-tune pretrained ViT-B/16 (5 epochs, ~5 min on T4)
!python train.py --model pretrained --epochs 5 --lr 1e-3 --device cuda

In [ ]:
# Print final accuracy for both models
import torch
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch import nn
from vit.data import download_data, create_dataloaders
from vit.model import ViT
from vit.utils import load_model

DATA_URL = 'https://github.com/mrdbourke/pytorch-deep-learning/releases/download/misc/pizza_steak_sushi.zip'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def get_accuracy(model, dataloader):
    model.to(device).train(False)
    correct = total = 0
    with torch.inference_mode():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(1) == y).sum().item()
            total += len(y)
    return correct / total * 100

image_path = download_data(source=DATA_URL, destination='pizza_steak_sushi')

# Custom ViT
_, test_dl, class_names = create_dataloaders(
    train_dir=image_path / 'train', test_dir=image_path / 'test',
    transform=transforms.Compose([
        transforms.Resize((224, 224)), transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]), batch_size=32)
custom_model = load_model(ViT(num_classes=3), 'models/custom_vit.pth')
custom_acc = get_accuracy(custom_model, test_dl)

# Pretrained ViT
weights = ViT_B_16_Weights.DEFAULT
_, test_dl_pre, _ = create_dataloaders(
    train_dir=image_path / 'train', test_dir=image_path / 'test',
    transform=weights.transforms(), batch_size=32)
pre_model = vit_b_16(weights=None)
pre_model.heads = nn.Linear(768, 3)
pre_model = load_model(pre_model, 'models/pretrained_vit.pth')
pre_acc = get_accuracy(pre_model, test_dl_pre)

print('=' * 42)
print('FINAL RESULTS')
print('=' * 42)
print(f'  Custom ViT from scratch (10ep): {custom_acc:.1f}%')
print(f'  Pretrained ViT-B/16   (5ep):   {pre_acc:.1f}%')
print('=' * 42)

In [ ]:
# Show loss curves
from IPython.display import Image as IPImage, display
print('Custom ViT loss curves:')
display(IPImage('models/custom_vit_loss_curves.png'))
print('Pretrained ViT loss curves:')
display(IPImage('models/pretrained_vit_loss_curves.png'))